# Leaf input and reduced-PID contract

This notebook audits data-compatible track/cluster inputs separately from MC
supervision. Fixture results test software behavior only.

## Setup and schema-v3 validation

In [ ]:
from pathlib import Path
import json, os, sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from hypertagging.data.notebook_fixtures import write_notebook_fixture_v4
from hypertagging.preprocessing.schema_v4 import load_payload_v4, SCHEMA_VERSION_V4
from hypertagging.preprocessing.pid_filter import PDG_TOKENS
from hypertagging.reconstruction.kinematics import track_energy_hypotheses
SEED = int(os.environ.get("HYPERTAGGING_NOTEBOOK_SEED", "20260730"))
torch.manual_seed(SEED); np.random.seed(SEED)
requested = os.environ.get("HYPERTAGGING_PARQUET", "").strip()
FIXTURE_MODE = not bool(requested)
INPUT = Path(requested) if requested else Path("/tmp/hypertagging_leaf_pid_v4.parquet")
if FIXTURE_MODE: write_notebook_fixture_v4(INPUT)
payload = load_payload_v4(INPUT)
if payload["schema_version"] != SCHEMA_VERSION_V4: raise ValueError("schema-v4 adaptation failed")
OUT = Path(os.environ.get("HYPERTAGGING_FIGURE_DIR", "/tmp/hypertagging_figures/leaf_pid"))
OUT.mkdir(parents=True, exist_ok=True)
print("TINY FIXTURE — NOT REAL DATA" if FIXTURE_MODE else "REAL PREPROCESSED SAMPLE")

## Raw p3, canonical input energy, and e/mu/pi/K/p hypotheses

In [ ]:
tracks = [node for event in payload["events"] for node in event["nodes"] if node["node_kind"] == "track"]
rows = []
for node in tracks:
    p3 = torch.tensor([node["px"], node["py"], node["pz"]], dtype=torch.float64)
    hypotheses = track_energy_hypotheses(p3).tolist()
    rows.append({
        "node_id": node["node_id"], "px": node["px"], "py": node["py"], "pz": node["pz"],
        "canonical_energy": node["reconstructed_energy"],
        **dict(zip(["E_e", "E_mu", "E_pi", "E_K", "E_p"], hypotheses)),
        "reco_charge": node["reco_charge"], "truth_charge": node["truth_charge"],
        "input_pid_token": node["input_pid_token"], "truth_pid_token": node["truth_pid_token"],
        "leaf_mode": node["leaf_kinematics_mode"],
    })
frame = pd.DataFrame(rows)
display(frame)
assert frame.input_pid_token.between(0, len(PDG_TOKENS)-1).all()
frame.to_csv(OUT / "leaf_pid_token_range.csv", index=False)
frame[["E_e","E_mu","E_pi","E_K","E_p"]].plot.hist(alpha=.45, bins=15, figsize=(9,5))
plt.title("Data-independent track energy hypotheses"); plt.tight_layout()
plt.savefig(OUT / "track_energy_hypotheses.png"); plt.show()

## PID likelihood availability and input/target separation

In [ ]:
availability = pd.DataFrame([node["pid_likelihood_availability"] for node in tracks])
display(availability)
display(pd.crosstab(frame.input_pid_token, frame.truth_pid_token, margins=True))
fallback_rate = float((frame.input_pid_token == 0).mean())
print("Unknown input PID fallback rate:", fallback_rate)
availability.mean().plot.bar(title="PIDLikelihood availability fraction")
plt.tight_layout(); plt.savefig(OUT / "pid_likelihood_availability.png"); plt.show()

## Explicit MC-present/MC-absent leakage check

In [ ]:
p3 = torch.tensor([[0.3, -0.2, 0.4]])
before = track_energy_hypotheses(p3)
changed_truth_pid = 8
after = track_energy_hypotheses(p3)
leakage_pass = bool(torch.equal(before, after))
report = {
    "no_truth_leakage_pass": leakage_pass,
    "changed_truth_pid_token": changed_truth_pid,
    "input_pid_token": 0,
    "canonical_hypothesis": "pion",
    "mc_absent_input_identical": leakage_pass,
}
(OUT / "leaf_input_leakage_check.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print(report)
assert leakage_pass

## Optional trained leaf PID head

In [ ]:
checkpoint = os.environ.get("HYPERTAGGING_CHECKPOINT", "")
print("No checkpoint supplied; PID confusion matrix is diagnostic-only." if not checkpoint else checkpoint)
confusion = pd.crosstab(frame.truth_pid_token, frame.input_pid_token)
display(confusion)
plt.imshow(confusion.to_numpy(), cmap="Blues"); plt.title("Input versus truth PID token")
plt.colorbar(); plt.tight_layout(); plt.savefig(OUT / "pid_confusion.png"); plt.show()